# 03 · Gradients: what the derivative of a simulation is, and how far to trust it

A simulation is a function: optical parameters in (how far light travels in water before it is absorbed, how
often it scatters, how much the black walls reflect), light per PMT out. Its **derivative** answers: *if this
parameter changed a little, how much would each PMT's light change?* That is what a fit needs to know which way to
move, and what tells you how well data can pin a parameter down.

lumen computes derivatives by **automatic differentiation** (AD): every arithmetic step of the simulation also
carries how its result moves with the parameter (the chain rule, applied by the computer), so one run gives the
light *and* its derivative. No finite-difference guessing, and no extra noise.

This notebook: one derivative; a check against finite differences; the best precision any measurement could reach
(the Cramér-Rao bound); a loss landscape over two parameters, and two ways of walking down it.

In [ ]:
import os
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np

import lumen
from lumen import tools
from lumen.compare import references
from lumen.tools import plot

WORK = Path(os.environ.get("LUMEN_NOTEBOOKS", "lumen-notebooks"))
WORK.mkdir(parents=True, exist_ok=True)
path = WORK / "sk.lumen"
if not path.exists():
    lumen.convert("lumen_wc:sk", out=path, data=references.store())
det = lumen.load(path)
p0 = det.params()

## A calibration flash

A light source at the tank's centre flashes ten million 400 nm photons in all directions (like the light-injection
systems detectors use to calibrate). The **model** is a plain function: parameters in, expected photoelectrons per
PMT out. Its random numbers are fixed (`seed=0`) and every random choice is made at the nominal values
(`reference=p0`; the light at other values is then weighted, not redrawn), so it is a smooth function of the
parameters: what a derivative and a fit need.

In [ ]:
flash = lumen.sources.Point((0.0, 0.0, 0.0), n_photons=1e7, lam=400.0)
SAMPLES = 2**17


def model(p):
    """Expected photoelectrons per PMT for the parameters p."""
    return lumen.readout.pe(lumen.simulate(det, flash, p, samples=SAMPLES, seed=0, reference=p0), p)


ABS, RAY, REFL = "materials/Water/ABSLENGTH", "materials/Water/RAYLEIGH", "surfaces/BSSkinSurface/REFLECTIVITY"

## One derivative

Scale the water's absorption length (every value of its table) by a factor `s`, and ask for the total
photoelectrons and their derivative at `s = 1`. `jax.jvp` runs the model once and returns both.

In [ ]:
def total_pe(s):
    return model(p0.scale(ABS, s)).sum()


value, slope = jax.jvp(total_pe, (1.0,), (1.0,))
print(f"total {float(value):.0f} PE; d(total)/d(factor) = {float(slope):.0f} PE")
print(f"so a 1 % longer absorption length changes the total by about {0.01 * float(slope):+.0f} PE "
      f"({float(slope) / float(value):+.3f} % per %)")

## Check it: automatic differentiation against finite differences

A finite difference estimates the slope from two nearby runs, (f(s + h) - f(s - h)) / 2h. Both runs use the same
random numbers, so the difference is not swamped by noise. `tools.check_gradient` does this for every PMT over
several seeds and tests whether the two agree within their errors (|z| <= 3 for the total, and the PMTs' spread as
expected by chance).

In [ ]:
def per_sensor(s, seed):
    p = p0.scale(ABS, s)
    return lumen.readout.pe(lumen.simulate(det, flash, p, samples=2**16, seed=seed, reference=p0), p)


report = tools.check_gradient(per_sensor, 1.0, seeds=8)
print(report)

## How well could a measurement do? The Cramér-Rao bound

Each PMT's count is a Poisson number: it fluctuates by about its square root. The **Fisher information** adds up,
over PMTs, how strongly each parameter moves each PMT's expectation compared with that noise:
F = Jᵀ diag(1/μ) J, where J holds the derivatives (one column per parameter) and μ the expected counts. Its
inverse is the **Cramér-Rao bound**: the smallest covariance any unbiased fit to such data can reach. It needs no
data, only derivatives.

A `tools.Space` names the parameters to vary; here each is one factor on its whole table.

In [ ]:
space = tools.Space(p0, factors=[ABS, RAY, REFL])
x1 = space.vector()  # all factors 1 (stored as their logarithms)
crb = np.linalg.inv(tools.fisher(model, space, x1))
bound = space.sigma(crb)
corr = crb / np.sqrt(np.outer(np.diag(crb), np.diag(crb)))
for k in space.keys:
    print(f"{k:40s} best possible precision ±{100 * bound[k]:.2f} % (one flash)")
print("correlations:\n", np.round(corr, 2))

## A loss landscape over two parameters

Make pretend data: the model at "true" factors (absorption 1.1, scattering 0.9), with Poisson noise. The **loss**
is the Poisson negative log-likelihood of that data: lowest where the model fits best. `tools.scan` evaluates it on
a grid, with its derivative along each axis by AD (and by finite differences between grid points, to compare).

In [ ]:
plane = tools.Space(p0, factors=[ABS, RAY])
truth = np.log([1.1, 0.9])
data = np.random.default_rng(1).poisson(np.asarray(model(plane.params(truth))))


def loss(p):
    return tools.poisson_nll(model(p), data)


line = tools.scan(loss, plane, {ABS: np.linspace(0.9, 1.3, 11)}, x0=truth)
plot.scan(line, space=plane, truth=truth)
plt.show()

Top: the loss along the absorption factor (the scattering factor at its true value). Bottom: its derivative from
AD (line) and from finite differences of the scanned points (circles); they agree, and the derivative crosses
zero at the minimum, next to the truth (dotted; not exactly at it, because the data are noisy).

Now both parameters. The white streamlines follow the downhill direction (minus the gradient).

In [ ]:
grid = tools.scan(loss, plane, {ABS: np.linspace(0.85, 1.35, 7), RAY: np.linspace(0.7, 1.1, 7)})

## Walking downhill: first order against second order

Two ways to find the minimum from the nominal values (both factors 1):

- **first order** (`tools.descend`): small steps along the gradient (the Adam method, which adapts the step to each
  parameter). It only knows the slope, so in a long tilted valley it zig-zags and needs many steps;
- **second order** (`tools.fit`, Fisher scoring): it also uses the Fisher information as the curvature, which turns
  the valley into a round bowl, and gets there in a few steps.

In [ ]:
first = tools.descend(model, plane, data, steps=30, lr=0.02)
second = tools.fit(model, plane, data, steps=8)
print(f"first order: {len(first.path) - 1} steps, factors {np.round(np.exp(first.x), 4)}")
print(f"second order: {len(second.path) - 1} steps, factors {np.round(np.exp(second.x), 4)}")
plot.scan(grid, space=plane, truth=truth, paths={"first order (Adam)": first, "second order (Fisher scoring)": second})
plt.show()

**Next:** 04 calibrates three parameters this way and checks the quoted uncertainties.